In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import json
import time
import base64
import traceback
from tqdm import tqdm
from openai import OpenAI
from google.colab import userdata

image_folder = '/content/drive/MyDrive/allRotatedPhotos'
output_json = '/content/drive/MyDrive/analys_masks.json'
token_log_path = '/content/drive/MyDrive/token_usage_summary.txt'
valid_extensions = ('.jpg', '.jpeg', '.png')

extraction_prompt = """
Extract all text from this turbocharger nameplate.
Maintain the line structure.
If you see serial numbers or part numbers, ensure they are exact.
CRITICAL: The text on this nameplate consists ONLY of Latin letters, numbers, hyphens (-), and slashes (/). You are strictly forbidden from outputting any other symbols like cyrillic letters or punctuation.
Return ONLY the extracted text, no introductory sentences.
"""

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

print("Connecting to OpenAI...")
api_key = userdata.get('GPT')
client = OpenAI(api_key=api_key)

os.makedirs(os.path.dirname(output_json), exist_ok=True)
os.makedirs(os.path.dirname(token_log_path), exist_ok=True)

if os.path.exists(output_json):
    os.remove(output_json)
    print("🗑️ Deleted existing JSON file. Starting fresh!")

results = {}

total_session_input_tokens = 0
total_session_output_tokens = 0

try:
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(valid_extensions)]
    print(f"Found {len(image_files)} images total. Starting extraction...\n")
except FileNotFoundError:
    print(f"Error: Could not find the folder {image_folder}. Please check the path.")
    image_files = []

if image_files:
    for filename in tqdm(image_files, desc="Processing Images"):
        image_path = os.path.join(image_folder, filename)

        mime_type = "image/png" if filename.lower().endswith('.png') else "image/jpeg"

        try:
            base64_image = encode_image(image_path)

            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": extraction_prompt
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:{mime_type};base64,{base64_image}",
                                    "detail": "high"
                                }
                            }
                        ]
                    }
                ],
                max_tokens=300,
                temperature=0.1
            )
            input_token_count = response.usage.prompt_tokens
            output_token_count = response.usage.completion_tokens
            total_session_input_tokens += input_token_count
            total_session_output_tokens += output_token_count

            extracted_text = response.choices[0].message.content.strip()
            results[filename] = extracted_text

            with open(output_json, 'w', encoding='utf-8') as json_file:
                json.dump(results, json_file, indent=4, ensure_ascii=False)

        except Exception as e:
            tqdm.write(f"❌ Error processing {filename}: {e}")
            results[filename] = f"ERROR: {str(e)}"

            with open(output_json, 'w', encoding='utf-8') as json_file:
                json.dump(results, json_file, indent=4, ensure_ascii=False)

        time.sleep(1)

    print(f"\n✅ Extraction complete! All data saved to: {output_json}")

    grand_total_tokens = total_session_input_tokens + total_session_output_tokens

    summary_text = (
        f"--- Token Usage Summary ---\n"
        f"Total Images Processed: {len(image_files)}\n"
        f"Total Input Tokens:     {total_session_input_tokens}\n"
        f"Total Output Tokens:    {total_session_output_tokens}\n"
        f"Grand Total Tokens:     {grand_total_tokens}\n"
        f"---------------------------\n"
    )

    print(f"\n{summary_text}")

    with open(token_log_path, 'w', encoding='utf-8') as f:
        f.write(summary_text)

    print(f"📄 Token usage summary successfully exported to: {token_log_path}")

In [ ]:
import os
import json
import time
import base64
import traceback
from tqdm import tqdm
from openai import OpenAI
from google.colab import userdata

image_folder = '/content/drive/MyDrive/allRotatedPhotos'
output_json = '/content/drive/MyDrive/analys_masks.json'
token_log_path = '/content/drive/MyDrive/token_usage_summary.txt'
valid_extensions = ('.jpg', '.jpeg', '.png')

extraction_prompt = """
Analyze this Garrett turbocharger nameplate and extract the information into a strict JSON format.

Follow these two rules:
1. "raw_text": Extract all visible text from the nameplate. Keep it exact, with no introductory text, descriptions, or added formatting. The text on this nameplate consists ONLY of Latin letters, numbers, hyphens (-), and slashes (/). You are strictly forbidden from outputting any other symbols ike cyrillic letters or punctuation.

2. "part_number": Extract ONLY the Garrett Part Number. It must follow the format of 6 digits, a hyphen (-), and 1 to 4 digits (e.g., 758219-3).
It MUST start with the digit 4, 7, or 8. If you cannot find a matching sequence, return null.

CRITICAL: Return ONLY valid JSON. Do not include markdown backticks (```) or any other conversational text.
Expected format:
{
  "raw_text": "all the text from the plate goes here...",
  "part_number": "XXXXXX-X..."
}
"""

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

print("Connecting to OpenAI...")
api_key = userdata.get('GPT')
client = OpenAI(api_key=api_key)

os.makedirs(os.path.dirname(output_json), exist_ok=True)
os.makedirs(os.path.dirname(token_log_path), exist_ok=True)

if os.path.exists(output_json):
    os.remove(output_json)
    print("🗑️ Deleted existing JSON file. Starting fresh!")

results = {}

total_session_input_tokens = 0
total_session_output_tokens = 0

try:
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(valid_extensions)]
    print(f"Found {len(image_files)} images total. Starting extraction...\n")
except FileNotFoundError:
    print(f"Error: Could not find the folder {image_folder}. Please check the path.")
    image_files = []

if image_files:
    for filename in tqdm(image_files, desc="Processing Images"):
        image_path = os.path.join(image_folder, filename)

        mime_type = "image/png" if filename.lower().endswith('.png') else "image/jpeg"

        try:
            base64_image = encode_image(image_path)

            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": extraction_prompt
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:{mime_type};base64,{base64_image}",
                                    "detail": "high" # Ensures OCR gets a high-res crop
                                }
                            }
                        ]
                    }
                ],
                max_tokens=300,
                temperature=0.1
            )

            input_token_count = response.usage.prompt_tokens
            output_token_count = response.usage.completion_tokens
            total_session_input_tokens += input_token_count
            total_session_output_tokens += output_token_count

            extracted_text = response.choices[0].message.content.strip()
            results[filename] = extracted_text

            with open(output_json, 'w', encoding='utf-8') as json_file:
                json.dump(results, json_file, indent=4, ensure_ascii=False)

        except Exception as e:
            tqdm.write(f"❌ Error processing {filename}: {e}")
            results[filename] = f"ERROR: {str(e)}"

            with open(output_json, 'w', encoding='utf-8') as json_file:
                json.dump(results, json_file, indent=4, ensure_ascii=False)

        time.sleep(1)

    print(f"\n✅ Extraction complete! All data saved to: {output_json}")

    grand_total_tokens = total_session_input_tokens + total_session_output_tokens

    summary_text = (
        f"--- Token Usage Summary ---\n"
        f"Total Images Processed: {len(image_files)}\n"
        f"Total Input Tokens:     {total_session_input_tokens}\n"
        f"Total Output Tokens:    {total_session_output_tokens}\n"
        f"Grand Total Tokens:     {grand_total_tokens}\n"
        f"---------------------------\n"
    )

    print(f"\n{summary_text}")

    with open(token_log_path, 'w', encoding='utf-8') as f:
        f.write(summary_text)

    print(f"📄 Token usage summary successfully exported to: {token_log_path}")